# **DATA CLEANING FOR MODEL NOTEBOOK**

## Objectives

*   Define data cleaning steps for Classification model

## Inputs

* **Raw Dataset:** inputs/datasets/raw/hotel_bookings.csv

## Outputs

* Code for data cleaning pipeline

## Additional Comments

* An analysis of the quality of the data and a summary of the cleaning steps taken before analysis can be found in notebook 2 (02_eda_and_cleaning.ipynb).
* This notebook will focus on the data cleaning steps to include in the ML pipeline to prepare the data for model training.
* Since the model needs to cope with new hotel data in production and may require retraining on updated data in the future, the data cleaning pipeline will be designed to manage potential future cases, not just current problems in the data.
* The y-Data Profile Report can be viewed in notebook 2 and will not be included again here.

---

# Import Packages, Load Data and Drop Specified Columns

Imports

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

Load data
- Convert data types for `agent`, `company`, `is_canceled` and `is_repeated_guest`

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_canceled': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

---

# Pre-Split Data Cleaning Steps

To avoid data leakage, the following must be done:
- Drop `reservation_status` and `reservation_status_date`
- Remove duplicated records

The latter of these **MUST** be done **BEFORE** the data is split into train and test set. Therefore, these steps will be taken now.

Drop `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Remove duplicates

In [ ]:
df = df.value_counts(dropna=False).reset_index(name="record_count")
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

***Note:*** *This approach assumes that the `record_count` (number of duplicate bookings) is known at the time of receiving the booking and that `is_duplicate` is therefore also known. Since these may have predictive power when training the model, these will be added to the data at this stage before splitting the data.*

# Train Test Split

Now the duplicates have been removed, the data can be split into train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

---

# Useful Variables and Functions

Useful Variables

In [ ]:
num_features = X_train.select_dtypes('number').columns.to_list()
cat_features = X_train.select_dtypes('object').columns.to_list()

This function (written by Code Institute) is useful for analysing the effect of data cleaning on the distribution of variables.

In [ ]:
import seaborn as sns
sns.set(style="whitegrid")
import matplotlib.pyplot as plt

def DataCleaningEffect(df_original,df_cleaned,variables_applied_with_method):

  flag_count=1 # Indicate plot number
  
  # distinguish between numerical and categorical variables
  categorical_variables = df_original.select_dtypes(exclude=['number']).columns 

  # scan over variables, 
    # first on variables that you applied the method
    # if the variable is a numerical plot, a histogram if categorical plot a barplot
  for set_of_variables in [variables_applied_with_method]:
    print("\n=====================================================================================")
    print(f"* Distribution Effect Analysis After Data Cleaning Method in the following variables:")
    print(f"{set_of_variables} \n\n")
  

    for var in set_of_variables:
      if var in categorical_variables:  # it is categorical variable: barplot
        
        df1 = pd.DataFrame({"Type":"Original","Value":df_original[var]})
        df2 = pd.DataFrame({"Type":"Cleaned","Value":df_cleaned[var]})
        dfAux = pd.concat([df1, df2], axis=0)
        fig , axes = plt.subplots(figsize=(15, 5))
        sns.countplot(hue='Type', data=dfAux, x="Value",palette=['#432371',"#FAAE7B"])
        axes.set(title=f"Distribution Plot {flag_count}: {var}")
        plt.xticks(rotation=90)
        plt.legend() 

      else: # it is numerical variable: histogram

        fig , axes = plt.subplots(figsize=(10, 5))
        sns.histplot(data=df_original, x=var, color="#432371", label='Original', kde=True,element="step", ax=axes)
        sns.histplot(data=df_cleaned, x=var, color="#FAAE7B", label='Cleaned', kde=True,element="step", ax=axes)
        axes.set(title=f"Distribution Plot {flag_count}: {var}")
        plt.legend() 

      plt.show()
      flag_count+= 1

---

# Data Cleaning Steps For Pipeline

Since the model will be applied to new hotel data when making predictions, and the dataset used for training may be updated in the future, the methods used for cleaning the data in the pipeline will differ slightly from those used in analysis.

## Missing Values

Summarise missing data

In [ ]:
summary = X_train.isnull().sum()
summary[summary > 0]

Although only four variables currently contain missing values, the data-cleaning pipeline will be designed to manage potential future cases. 

### Categorical variables

Get categorical features

In [ ]:
cat_features

These are unlikely to have missing values:
- `hotel`, `arrival_date_month`

These features are required - impute with mode:
- `hotel`, `arrival_date_month`, `meal`, `market_segment`, `distribution_channel`, `reserved_room_type`, `assigned_room_type`, `deposit_type`, `customer_type`

Features for which missingness may be informative - impute with 'Missing':
- `country`, `agent`, `company`

Features which would benefit from a custom transformer:
- `is_repeated_guest` (can be imputed from `previous_bookings_not_cancelled`)

Make a copy of dataframe for testing transformations

In [ ]:
variables_engineering = cat_features.copy()
variables_engineering.append('previous_bookings_not_canceled')  # needed to impute is_repeated_guest

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

To test imputer, artificially add missing values

In [ ]:
row_idx = np.random.choice(df_engineering.index, size=2, replace=False)

# Change first of these rows - all missing values
for var in variables_engineering:
    df_engineering.loc[row_idx[0], var] = np.nan

# Change second of these rows - previous_bookings_not_canceled=1
df_engineering.loc[row_idx[1], 'is_repeated_guest'] = np.nan
df_engineering.loc[row_idx[1], 'previous_bookings_not_canceled'] = 1

df_engineering.loc[row_idx]

Define custom transformer for imputing `is_repeated_guest`

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RepeatedGuestImputer(BaseEstimator, TransformerMixin):
    def __init__(self, booking_col='previous_bookings_not_canceled', target_col='is_repeated_guest'):
        self.booking_col = booking_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Set a dummy attribute to signal fitted status (to remove warning)
        self.fitted_ = True
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isnull()
        X.loc[mask_missing, self.target_col] = (X.loc[mask_missing, self.booking_col] > 0).astype(int)
        return X


Define variables, transformers and pipeline

In [ ]:
from feature_engine.imputation import CategoricalImputer
from sklearn.pipeline import Pipeline


# Define variables
impute_missing_label_variables = ['country', 'company', 'agent']

impute_mode_variables = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]

# Imputer for 'Missing' label
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=impute_missing_label_variables,
)

# Imputer for mode
imputer_mode = CategoricalImputer(
    imputation_method="frequent",
    variables=impute_mode_variables
)

# Imputer for repeated guest
imputer_repeated_guest = RepeatedGuestImputer()

# pipeline
pipeline = Pipeline([
    ("imputer_missing_label", imputer_missing_label),
    ("imputer_mode", imputer_mode),
    ("imputer_repeated_guest", imputer_repeated_guest),
])

Fit transformer on df_engineering

In [ ]:
df_engineering = pipeline.fit_transform(df_engineering)
df_engineering

Check no missing values remain

***NOTE:*** *`previous_bookings_not_canceled` is not included in pipeline so may still have missing values*

In [ ]:
summary = df_engineering.isnull().sum()
summary[summary > 0]

Check rows that were artifically changed

In [ ]:
df_engineering.loc[row_idx]

The transformer worked correctly and will therefore be applied to the train and test sets

In [ ]:
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

### Numeric Variables

View numerical features

In [ ]:
num_features

These features should not have missing values since they are added to the dataset after loading:
- `record_count`, `is_duplicate`

These are unlikely to have missing values:
- `lead_time`, `arrival_date_year`, `arrival_date_week_number`, `arrival_date_day_of_month`, `stays_in_weekend_nights`, `stays_in_week_nights`, `days_in_waiting_list`

If these features have missing values, **impute with median**:
- `adr`, `adults`

If these features have missing values they likely imply absence, **so impute with zero**:
- `children`, `babies`, `previous_cancellations`, `previous_bookings_not_canceled`, `booking_changes`, `required_car_parking_spaces`, `total_of_special_requests`

***NOTE:*** *In the current dataset, out of all of these features only `children` has a missing value and only for one observation*

Make a copy of dataframe for testing transformations

In [ ]:
variables_engineering = [
    'adr', 'adults', 'children', 'babies','previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]

df_engineering = X_train[variables_engineering].copy()
df_engineering

To test imputer, artificially add missing values

In [ ]:
row_idx = np.random.choice(df_engineering.index, size=1)

# Change row - all missing values
for var in variables_engineering:
    df_engineering.loc[row_idx[0], var] = np.nan

df_engineering.loc[row_idx]

Define variables, transformers and pipeline

In [ ]:
from feature_engine.imputation import ArbitraryNumberImputer, MeanMedianImputer

# Define variables
impute_zero_variables = [
    'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]
impute_median_variables = ['adr', 'adults']

# Imputer for zero
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=impute_zero_variables
)

# Imputer for median
imputer_median = MeanMedianImputer(
    imputation_method="median",
    variables=impute_median_variables
)

# pipeline
pipeline = Pipeline([
    ("imputer_zero", imputer_zero),
    ("imputer_median", imputer_median)
])

Fit transformer on df_engineering

In [ ]:
df_engineering = pipeline.fit_transform(df_engineering)
df_engineering

Check no missing values remain

In [ ]:
summary = df_engineering.isnull().sum()
summary[summary > 0]

Check row that was artifically changed

In [ ]:
df_engineering.loc[row_idx]

The transformer worked correctly and will therefore be applied to the train and test sets

In [ ]:
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

## Undefined Values

In notebook 2, certain features (`meal`, `market_segment`, `distribution_channel`) were found to include a value of 'Undefined'. The decision was made to drop these for the purposes of analysing patterns in the data. However, this label may have predictive power when training the model so these observations will not be dropped as part of the data-cleaning pipeline.

**NO ACTION REQUIRED**

## Outliers

Outliers will be capped using business meaningful boundaries to prevent the data from being skewed. The y-Data Profile Report from workbook 2 was used to inform the choices for these values.

View numerical features

In [ ]:
num_features

Make a copy of dataframe for testing transformations

In [ ]:
variables_engineering = num_features.copy()

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

Define transformer for capping outliers

In [ ]:
from feature_engine.outliers import ArbitraryOutlierCapper

max_capping_dict = {
    'lead_time': 600,
    'arrival_date_year': 2017,
    'arrival_date_week_number': 53,
    'arrival_date_day_of_month': 31,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'adults': 4,
    'children': 4,
    'babies': 2,
    'previous_cancellations': 10,
    'previous_bookings_not_canceled': 20,
    'booking_changes': 10,
    'days_in_waiting_list': 60,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_year': 2015,
    'arrival_date_week_number': 1,
    'arrival_date_day_of_month': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'adults': 1,
    'children': 0,
    'babies': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'days_in_waiting_list': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Fit the transformer on `df_engineering`

In [ ]:
df_engineering = outlier_capper.fit_transform(df_engineering)
df_engineering.head(3)

Evaluate the effect of the transformer on the distribution of numeric variables

In [ ]:
DataCleaningEffect(X_train, df_engineering, num_features)

The transformation has not negatively affected the distributions so apply to train and test set.

In [ ]:
X_train = outlier_capper.fit_transform(X_train)
X_test = outlier_capper.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

---

# Summary of Data Cleaning Process

## Load Data

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_canceled': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

## Pre-Split Data Cleaning

Then drop columns `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Remove duplicates

df = df.value_counts(dropna=False).reset_index(name="record_count")
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

## Data Cleaning Pipeline

Define custom transformer for imputing `is_repeated_guest`

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RepeatedGuestImputer(BaseEstimator, TransformerMixin):
    def __init__(self, booking_col='previous_bookings_not_canceled', target_col='is_repeated_guest'):
        self.booking_col = booking_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Set a dummy attribute to signal fitted status (to remove warning)
        self.fitted_ = True
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isnull()
        X.loc[mask_missing, self.target_col] = (X.loc[mask_missing, self.booking_col] > 0).astype(int)
        return X


Define variables and transformers for using in the pipeline

In [ ]:
from feature_engine.imputation import (
    CategoricalImputer,
    ArbitraryNumberImputer,
    MeanMedianImputer,
)
from feature_engine.outliers import ArbitraryOutlierCapper

# Categorical imputer for 'Missing' label
impute_missing_label_variables = ['country', 'company', 'agent']
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=impute_missing_label_variables,
)

# Categorical imputer for mode
impute_mode_variables = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]
imputer_mode = CategoricalImputer(
    imputation_method="frequent",
    variables=impute_mode_variables
)

# Categorical imputer for repeated guest
imputer_repeated_guest = RepeatedGuestImputer()

# Numeric imputer for zero
impute_zero_variables = [
    'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=impute_zero_variables
)

# Numeric imputer for median
impute_median_variables = ['adr', 'adults']
imputer_median = MeanMedianImputer(
    imputation_method="median",
    variables=impute_median_variables
)

# Outlier capper
max_capping_dict = {
    'lead_time': 600,
    'arrival_date_year': 2017,
    'arrival_date_week_number': 53,
    'arrival_date_day_of_month': 31,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'adults': 4,
    'children': 4,
    'babies': 2,
    'previous_cancellations': 10,
    'previous_bookings_not_canceled': 20,
    'booking_changes': 10,
    'days_in_waiting_list': 60,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_year': 2015,
    'arrival_date_week_number': 1,
    'arrival_date_day_of_month': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'adults': 1,
    'children': 0,
    'babies': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'days_in_waiting_list': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Define data cleaning pipeline

In [ ]:
from sklearn.pipeline import Pipeline

data_cleaning_pipeline = Pipeline([
    ("imputer_missing_label", imputer_missing_label),
    ("imputer_mode", imputer_mode),
    ("imputer_repeated_guest", imputer_repeated_guest),
    ("imputer_zero", imputer_zero),
    ("imputer_median", imputer_median),
    ("outlier_capper", outlier_capper),
])

Fit the pipeline and transform data.

In [ ]:
X_train = data_cleaning_pipeline.fit_transform(X_train)
X_test = data_cleaning_pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

# Conclusions and Next Steps


In this notebook, the steps required for cleaning the data before model training were considered

**Pre-split data cleaning steps** include:
- Changing data types *(when loading the data)*
  - `agent`, `company`, `is_canceled` and `is_repeated_guest` all changed to type `object`
- Dropping features that would cause data leakage
  - `reservation_status` and `reservation_status_date`
- Aggregating duplicate records
  - summarise deduplication by adding new features: `record_count` and `is_duplicate`

The **data cleaning pipeline** will include:

- Impute missing values for categorical features
  - Impute 'Missing' label: `country`, `company`, `agent`
  - Impute mode: `hotel`, `arrival_date_month`, `meal`, `market_segment`,
    `distribution_channel`, `reserved_room_type`, `assigned_room_type`,
    `deposit_type`, `customer_type`
  - Customer transformer `RepeatedGuestImputer`: uses `previous_bookings_not_canceled` to impute value for `is_repeated_guest`

- Impute missing values for numeric features
  - Impute median: `adr`, `adults`
  - Impute with zero: `children`, `babies`, `previous_cancellations`, `previous_bookings_not_canceled`, `booking_changes`, `required_car_parking_spaces`, `total_of_special_requests`

- Cap Outliers using boundaries specified below

  - **max_capping_dict** = {
    `lead_time`: 600,
    `arrival_date_year`: 2017,
    `arrival_date_week_number`: 53,
    `arrival_date_day_of_month`: 31,
    `stays_in_weekend_nights`: 10,
    `stays_in_week_nights`: 25,
    `adults`: 4,
    `children`: 4,
    `babies`: 2,
    `previous_cancellations`: 10,
    `previous_bookings_not_canceled`: 20,
    `booking_changes`: 10,
    `days_in_waiting_list`: 60,
    `adr`: 400,
    `required_car_parking_spaces`: 2,
    `total_of_special_requests`: 5
  }

  - **min_capping_dict** = {
    `lead_time`: 1,
    `arrival_date_year`: 2015,
    `arrival_date_week_number`: 1,
    `arrival_date_day_of_month`: 1,
    `stays_in_weekend_nights`: 0,
    `stays_in_week_nights`: 0,
    `adults`: 1,
    `children`: 0,
    `babies`: 0,
    `previous_cancellations`: 0,
    `previous_bookings_not_canceled`: 0,
    `booking_changes`: 0,
    `days_in_waiting_list`: 0,
    `adr`: 0,
    `required_car_parking_spaces`: 0,
    `total_of_special_requests`: 0
  }

***NOTE:*** *Values of 'Undefined' are suitable for model training so will ke kept unchanged.*

The next notebook will consider the possible feature engineering steps to consider to optimise model training.